In [ ]:
# installation
print("CLEAN INSTALL")
import sys

print(" Uninstalling broken libraries...")
!{sys.executable} -m pip uninstall -y pillow numpy scipy scikit-learn matplotlib seaborn ultralytics
print("Installing clean, compatible versions...")
!{sys.executable} -m pip install "numpy==1.26.4" "Pillow==10.4.0" "scipy" "scikit-learn" "matplotlib" "seaborn" "ultralytics"

print("\n All packages ready!")

In [ ]:

# IMPORTS

import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, 
                            confusion_matrix, classification_report, roc_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Device: {device}\n")


In [ ]:
import numpy as np
from scipy import signal
from scipy.ndimage import uniform_filter1d
from scipy.signal import find_peaks

# HELPER FUNCTIONS: SYNC & SEGMENTATION

def smooth(data, w=9):
    """Moving average smoothing"""
    if len(data) < w: return data
    return np.convolve(data, np.ones(w)/w, mode='same')

def sync_signals(s1, s2):
    """Synchronize Front and Side views using Hip movement"""
    if len(s1) == 0 or len(s2) == 0: return 0
    
    # Normalize signals
    s1 = (s1 - np.mean(s1)) / (np.std(s1) + 1e-6)
    s2 = (s2 - np.mean(s2)) / (np.std(s2) + 1e-6)
    
    # Cross-correlation to find time shift
    correlation = signal.correlate(s1, s2, mode='full')
    lags = signal.correlation_lags(len(s1), len(s2), mode='full')
    lag = lags[np.argmax(correlation)]
    return int(lag)

def segment_force_5(keypoints_sequence):
    """
    Forces the sequence into exactly 5 segments.
    1. Tries to find 5 peaks (bottom of squats).
    2. If that fails, chops the video into 5 equal time chunks.
    """
    # Extract Hip Y-trajectory
    hip_y = [(kp[11][1] + kp[12][1]) / 2 for kp in keypoints_sequence]
    hip_y = np.array(hip_y)
    
    if np.max(hip_y) - np.min(hip_y) < 0.01:
        return np.array_split(keypoints_sequence, 5)

    # Normalize and Smooth
    hip_norm = (hip_y - np.min(hip_y)) / (np.max(hip_y) - np.min(hip_y))
    hip_smooth = smooth(hip_norm, 15)
    peaks, _ = find_peaks(hip_smooth, distance=30, prominence=0.05)
    
    segments = []
    
    if len(peaks) == 5:
        duration = len(keypoints_sequence) // 5
        half_win = duration // 2
        
        for p in peaks:
            start = max(0, p - half_win)
            end = min(len(keypoints_sequence), p + half_win)
            segments.append(keypoints_sequence[start:end])
    else:
        
        segments = np.array_split(keypoints_sequence, 5)
    
    return segments

print("Helper Functions Ready")

In [ ]:

# SETUP

for d in ['/kaggle/working/data/processed/keypoints',
          '/kaggle/working/data/processed/labels',
          '/kaggle/working/models',
          '/kaggle/working/results']:
    os.makedirs(d, exist_ok=True)

base = '/kaggle/input/squat-prediction/raw_videos'
pairs = []
for f in sorted(os.listdir(f'{base}/front_view')):
    if f.endswith('.mp4'):
        pid = f.replace('person_', '').replace('.mp4', '')
        if os.path.exists(f'{base}/side_view/side_person{pid}.mp4'):
            pairs.append({
                'pid': pid,
                'f': f'{base}/front_view/{f}',
                's': f'{base}/side_view/side_person{pid}.mp4'
            })

print(f"Found {len(pairs)} pairs → {len(pairs)*5} squats expected\n")


In [ ]:

# LABELER 

class Labeler:
    def angle(self, p1, p2, p3):
        v1, v2 = p1[:2]-p2[:2], p3[:2]-p2[:2]
        return np.degrees(np.arccos(np.clip(
            np.dot(v1,v2)/(np.linalg.norm(v1)*np.linalg.norm(v2)+1e-6), -1, 1)))
    
    def label(self, kps):
        hips = [k[11][1] for k in kps]
        # Find the frame with the lowest hip position 
        bot_idx = np.argmax(hips)
        bot = kps[bot_idx]
        errs = []
        
        # 1. Depth: Widen range to 70-115 degrees
        if all(bot[i][2]>0.5 for i in [12,14,16]):
            ang = self.angle(bot[12], bot[14], bot[16])
            if not (70 <= ang <= 115):
                errs.append('depth')
        
        # 2. Knee Valgus: Threshold 0.6
        if all(bot[i][2]>0.5 for i in [13,14,15,16]):
            knee_dist = abs(bot[13][0]-bot[14][0])
            ank_dist = abs(bot[15][0]-bot[16][0])
            if knee_dist / (ank_dist + 1e-6) < 0.6: 
                errs.append('knee')
        
        # 3. Back Angle: Widen range to 30-85 degrees
        if all(bot[i][2]>0.5 for i in [6,12,14]):
            back_ang = self.angle(bot[6], bot[12], bot[14])
            if not (30 <= back_ang <= 85):
                errs.append('back')
        
        return 1 if len(errs)==0 else 0, errs

labeler = Labeler()
print(" Labeler Ready")

In [ ]:

# PROCESSOR

class Proc:
    def __init__(self):
        print("Loading YOLOv8...")
        self.m = YOLO('yolov8n-pose.pt')
        print(" YOLOv8 ready\n")
    
    def get_kps(self, video_path):
        """Extract all keypoints from a video"""
        cap = cv2.VideoCapture(video_path)
        kps = []
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            results = self.m(frame, verbose=False)
            if len(results[0].keypoints.data) > 0:
                # Get keypoints of the first person
                kps.append(results[0].keypoints.data[0].cpu().numpy())
            else:
                # Padding if no person found
                kps.append(kps[-1] if kps else np.zeros((17,3)))
                
        cap.release()
        return np.array(kps)
    
    def process(self, f_path, s_path):
        # 1. Extract Keypoints
        fk = self.get_kps(f_path)
        sk = self.get_kps(s_path)
        
        # Safety check
        if len(fk) < 10 or len(sk) < 10:
            return [], []

        # 2. Synchronize
        fsig = [(k[11][1] + k[12][1])/2 for k in fk]
        ssig = [(k[11][1] + k[12][1])/2 for k in sk]
        
        offset = sync_signals(fsig, ssig)
        
        # Trim to match timing
        if offset > 0:
            fk = fk[offset:]
            sk = sk[:len(fk)]
        elif offset < 0:
            sk = sk[abs(offset):]
            fk = fk[:len(sk)]
            
        min_len = min(len(fk), len(sk))
        fk = fk[:min_len]
        sk = sk[:min_len]
        
        s_segs = segment_force_5(sk)
        f_segs = np.array_split(fk, 5)
        return f_segs, s_segs

proc = Proc()

In [ ]:

# PROCESS VIDEOS

print("PROCESSING (Target: 100 Squats)")
data = []
for p in tqdm(pairs, desc="Videos"):
    try:
        # Process video pair
        f_segs, s_segs = proc.process(p['f'], p['s'])
        count = len(f_segs)
        print(f"Person {p['pid']}: {count} squats")
        
        for i in range(count):
            fs = f_segs[i]
            ss = s_segs[i]
            
            # Label based on Side View
            label, errs = labeler.label(ss)
            sid = f"p{p['pid']}_s{i+1}"
            
            # Save Keypoints
            np.save(f'/kaggle/working/data/processed/keypoints/{sid}_f.npy', fs)
            np.save(f'/kaggle/working/data/processed/keypoints/{sid}_s.npy', ss)
            
            # Save Meta
            d = {'sid': sid, 'pid': p['pid'], 'label': int(label), 
                 'errors': errs, 'frames': len(fs)}
            
            with open(f'/kaggle/working/data/processed/labels/{sid}.json', 'w') as f:
                json.dump(d, f)
            data.append(d)
            
    except Exception as e:
        print(f" Error Person {p['pid']}: {e}")

print(f"\n Total Squats Extracted: {len(data)}")

In [ ]:

# SUMMARY
corr = sum(d['label'] for d in data)
print(f"DATASET: {len(data)} squats")
print(f"   Correct: {corr}")
print(f"   Incorrect: {len(data)-corr}")

fig, ax = plt.subplots(1,2,figsize=(12,4))
ax[0].pie([corr, len(data)-corr], labels=['Correct','Incorrect'],
          colors=['#2ecc71','#e74c3c'], autopct='%1.1f%%')
ax[0].set_title('Labels')

errs = {}
for d in data:
    for e in d['errors']:
        errs[e] = errs.get(e,0) + 1
if errs:
    ax[1].barh(list(errs.keys()), list(errs.values()), color='#e74c3c')
    ax[1].set_title('Errors')
plt.tight_layout()
plt.savefig('/kaggle/working/results/dataset.png', dpi=150)
plt.show()


In [ ]:

# MODEL

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.LSTM(51, 128, 2, batch_first=True, dropout=0.3, bidirectional=True)
        self.s = nn.LSTM(51, 128, 2, batch_first=True, dropout=0.3, bidirectional=True)
        self.attn = nn.MultiheadAttention(256, 4, dropout=0.3, batch_first=True)
        self.fuse = nn.Sequential(nn.Linear(512,256), nn.LayerNorm(256), nn.ReLU(), nn.Dropout(0.3))
        self.temp = nn.Sequential(nn.Linear(256,128), nn.Tanh(), nn.Linear(128,1))
        self.cls = nn.Sequential(nn.Linear(256,128), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64,2))
    
    def forward(self, f, s):
        fo, _ = self.f(f)
        so, _ = self.s(s)
        fa, _ = self.attn(fo, so, so)
        fused = self.fuse(torch.cat([fa, so], -1))
        aw = F.softmax(self.temp(fused), 1)
        return self.cls(torch.sum(aw * fused, 1))


In [ ]:

# DATASET (WITH NORMALIZATION & AUGMENTATION)

class DS(Dataset):
    def __init__(self, data, seq=60, aug=False):
        self.data = data
        self.seq = seq
        self.aug = aug
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, i):
        d = self.data[i]
        # Load raw keypoints
        f = np.load(f'/kaggle/working/data/processed/keypoints/{d["sid"]}_f.npy')
        s = np.load(f'/kaggle/working/data/processed/keypoints/{d["sid"]}_s.npy')
        
        #  AUGMENTATION (Training Only) 
        if self.aug:
            # 1. Horizontal Flip 
            if np.random.rand() > 0.5:
                for kp in [f, s]:
                    for l, r in [(1,2), (3,4), (5,6), (7,8), (9,10), (11,12), (13,14), (15,16)]:
                        kp[:, l, :], kp[:, r, :] = kp[:, r, :].copy(), kp[:, l, :].copy()
            
            # 2. Random Jitter (Noise)
            if np.random.rand() > 0.3:
                noise_f = np.random.normal(0, 0.01, f.shape) 
                noise_s = np.random.normal(0, 0.01, s.shape)
                noise_f[:, :, 2] = 0
                noise_s[:, :, 2] = 0
                f = f + noise_f
                s = s + noise_s
        
        # PRE-PROCESSING 
        return (torch.FloatTensor(self._p(f)), 
                torch.FloatTensor(self._p(s)), 
                torch.LongTensor([d['label']]))
    
    def _p(self, k):
        """Normalize and Pad Sequence"""
        for frame_idx in range(k.shape[0]):
            # Calculate hip center (x, y)
            hip_x = (k[frame_idx, 11, 0] + k[frame_idx, 12, 0]) / 2
            hip_y = (k[frame_idx, 11, 1] + k[frame_idx, 12, 1]) / 2
            
            # Subtract hip center from all x, y coordinates
            k[frame_idx, :, 0] -= hip_x
            k[frame_idx, :, 1] -= hip_y

        # 2. Flatten (Frames, 51)
        k_flat = k.reshape(k.shape[0], -1)
        
        # 3. Pad or Trim to fixed sequence length
        if k_flat.shape[0] < self.seq:
            # Pad with zeros if too short
            pad = np.zeros((self.seq - k_flat.shape[0], k_flat.shape[1]))
            k_flat = np.vstack([k_flat, pad])
        elif k_flat.shape[0] > self.seq:
            # Center crop if too long
            st = (k_flat.shape[0] - self.seq) // 2
            k_flat = k_flat[st:st + self.seq]
            
        return k_flat

# Prepare Data Loaders
pids = list(set(d['pid'] for d in data))
tr_p, va_p = train_test_split(pids, test_size=0.2, random_state=42)

tr_data = [d for d in data if d['pid'] in tr_p]
va_data = [d for d in data if d['pid'] in va_p]

print(f"Train: {len(tr_data)} squats | Val: {len(va_data)} squats\n")
tr_ld = DataLoader(DS(tr_data, aug=True), batch_size=4, shuffle=True)
va_ld = DataLoader(DS(va_data, aug=False), batch_size=4)

In [ ]:

# TRAINING (WITH CLASS WEIGHTS)

print("TRAINING")
model = Model().to(device)

# CALCULATE CLASS WEIGHTS 
n_incorrect = sum(1 for d in tr_data if d['label'] == 0)
n_correct = sum(1 for d in tr_data if d['label'] == 1)
n_incorrect = max(n_incorrect, 1)
n_correct = max(n_correct, 1)

weight_correct = n_incorrect / n_correct
class_weights = torch.FloatTensor([1.0, weight_correct]).to(device)

print(f"Class Counts -> Incorrect: {n_incorrect}, Correct: {n_correct}")
print(f"Class Weights -> Incorrect: 1.0, Correct: {weight_correct:.2f}\n")

crit = nn.CrossEntropyLoss(weight=class_weights)
opt = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-3) 
sch = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=10)

best, pat = 0, 0
hist = {'tl':[], 'ta':[], 'vl':[], 'va':[]}

for ep in range(1, 151): 
    # Train
    model.train()
    tl, tc, tt = 0, 0, 0
    for f, s, l in tr_ld:
        f, s, l = f.to(device), s.to(device), l.squeeze().to(device)
        opt.zero_grad()
        out = model(f, s)
        loss = crit(out, l)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        
        tl += loss.item()
        tc += (torch.argmax(out,1)==l).sum().item()
        tt += l.size(0)
    
    ta = 100*tc/tt
    tl /= len(tr_ld)
    
    # Validate
    model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for f, s, l in va_ld:
            f, s, l = f.to(device), s.to(device), l.squeeze().to(device)
            out = model(f, s)
            vl += crit(out, l).item()
            vc += (torch.argmax(out,1)==l).sum().item()
            vt += l.size(0)
    
    va = 100*vc/vt
    vl /= len(va_ld)
    
    sch.step(va)
    hist['tl'].append(tl)
    hist['ta'].append(ta)
    hist['vl'].append(vl)
    hist['va'].append(va)
    
    if ep%10==0 or ep==1:
        print(f"Ep {ep:3d} | Train: {ta:.1f}% | Val: {va:.1f}% | Loss: {tl:.4f}")
    
    if va > best:
        best, pat = va, 0
        torch.save({'m': model.state_dict(), 'a': va}, '/kaggle/working/models/best.pth')
        if ep > 5:
            print(f" New Best: {va:.1f}%")
    else:
        pat += 1
    
    if pat >= 25: 
        print(f"\nEarly stop at {ep}")
        break

print(f"\n Best Val Accuracy: {best:.2f}%")

In [ ]:

# PLOTS & EVAL

fig, ax = plt.subplots(1,2,figsize=(14,5))
ax[0].plot(hist['tl'], label='Train', marker='o', ms=3)
ax[0].plot(hist['vl'], label='Val', marker='s', ms=3)
ax[0].set_title('Loss')
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(hist['ta'], label='Train', marker='o', ms=3)
ax[1].plot(hist['va'], label='Val', marker='s', ms=3)
ax[1].set_title('Accuracy')
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/results/training.png', dpi=150)
plt.show()

ckpt = torch.load('/kaggle/working/models/best.pth')
model.load_state_dict(ckpt['m'])
model.eval()

preds, labels, probs = [], [], []
with torch.no_grad():
    for f, s, l in va_ld:
        f, s, l = f.to(device), s.to(device), l.squeeze().to(device)
        out = model(f, s)
        pr = F.softmax(out, 1)
        preds.extend(torch.argmax(out,1).cpu().numpy())
        labels.extend(l.cpu().numpy())
        probs.extend(pr.cpu().numpy())

print(classification_report(labels, preds, target_names=['Incorrect','Correct']))

cm = confusion_matrix(labels, preds)
fpr, tpr, _ = roc_curve(labels, np.array(probs)[:,1])
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(1,2,figsize=(14,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0],
            xticklabels=['Incorrect','Correct'],
            yticklabels=['Incorrect','Correct'])
ax[0].set_title('Confusion Matrix')

ax[1].plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC={roc_auc:.3f}')
ax[1].plot([0,1], [0,1], 'navy', lw=2, ls='--')
ax[1].set_title('ROC')
ax[1].legend()
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/results/eval.png', dpi=150)
plt.show()

print(" DONE!")
print(f"Accuracy: {best:.2f}%")
print(f"Squats: {len(data)}")
print("\nDownload: /kaggle/working/models/best.pth")
